# Scorecard Components Demo

Show per-variable point contributions and total score using `scorecard_components`.


In [ ]:
import pandas as pd
import polars as pl
from scorecardpl import split_df, var_filter, woebin, woebin_ply, scorecard, scorecard_components


## Load, split, bin, and train logistic scorecard

In [ ]:
uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
cols = [
    'Status', 'Duration', 'CreditHistory', 'Purpose', 'CreditAmount',
    'Savings', 'Employment', 'InstallmentRate', 'PersonalStatusSex', 'OtherDebtors',
    'ResidenceSince', 'Property', 'Age', 'OtherInstallmentPlans', 'Housing',
    'ExistingCredits', 'Job', 'Liables', 'Telephone', 'ForeignWorker', 'class'
]
df_pd = pd.read_csv(uci_url, sep=' ', header=None, names=cols)
df_pd['y'] = (df_pd['class'] == 2).astype(int)
df_pd = df_pd.drop(columns=['class'])
df = pl.from_pandas(df_pd)
y = 'y'
train, valid = split_df(df, y=y, test_size=0.3, random_state=42)
train = var_filter(train, y=y)
bins = woebin(train, y=y, x=[c for c in train.columns if c != y],
                bins=6, method='chi2', chi2_params={'init_bins': 60}, monotonic='auto', cat_max_bins=5)
train_w = woebin_ply(train, bins)
sc = scorecard(bins, y=y, data=train_w)


## Components for a few rows

In [ ]:
valid_w = woebin_ply(valid, bins)
comp = scorecard_components(valid_w.head(10), sc.points_map, include_intercept=True, total_col='score')
comp
